In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mitgandhi10/dataset-for-cnn")

print("Path to dataset files:", path)

100%|██████████| 85.0M/85.0M [00:06<00:00, 14.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mitgandhi10/dataset-for-cnn/versions/2


In [4]:
import os

DATA_DIR = path

print('isi folder dataset:', os.listdir(DATA_DIR))

isi folder dataset: ['xray_dataset_covid19']


In [6]:
!ls {DATA_DIR}


xray_dataset_covid19


In [8]:
cd {DATA_DIR}


/root/.cache/kagglehub/datasets/mitgandhi10/dataset-for-cnn/versions/2


In [9]:
!ls xray_dataset_covid19/

'single prediction'   test   train


In [12]:
import os
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

base_path = path

train_dir = os.path.join(base_path, 'train')
test_dir = os.path.join(base_path, 'test')

print("Train directory:", train_dir)
print("Test directory:", test_dir)

Train directory: /root/.cache/kagglehub/datasets/mitgandhi10/dataset-for-cnn/versions/2/train
Test directory: /root/.cache/kagglehub/datasets/mitgandhi10/dataset-for-cnn/versions/2/test


In [14]:
import os

# Adjust paths to include the 'xray_dataset_covid19' subdirectory
train_dir = os.path.join(path, 'xray_dataset_covid19', 'train')
test_dir = os.path.join(path, 'xray_dataset_covid19', 'test')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

#hyperparameter
num_epochs = 5
num_classes = 2
batch_size = 32
learning_rate = 0.001

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = torchvision.datasets.ImageFolder(root=train_dir, transform=transform)
test_dataset = torchvision.datasets.ImageFolder(root=test_dir, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Dataset berhasil dimuat! Label kelas: {train_dataset.classes}")
print("Train dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))

Using device: cuda
Dataset berhasil dimuat! Label kelas: ['NORMAL', 'PNEUMONIA']
Train dataset size: 148
Test dataset size: 40


In [17]:
class ConvNet(nn.Module):
  def __init__(self, num_classes=2):
    super(ConvNet, self).__init__()
    self.layer1 = nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2))
    self.layer2 = nn.Sequential(
        nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2))
    self.fc = nn.Linear(16 * 16 * 32, num_classes)  # Note: 64x64 input with two 2x2 poolings becomes 16x16

  def forward(self, x):
    out = self.layer1(x)
    out = self.layer2(out)
    out = out.reshape(out.size(0), -1)
    out = self.fc(out)
    return out

model = ConvNet(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

total_step = len(train_loader)
for epoch in range(num_epochs):
  for i, (images, labels) in enumerate(train_loader):
    images = images.to(device)
    labels = labels.to(device)

    outputs = model(images)
    loss = criterion(outputs, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (i+1) % 100 == 0:
      print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{total_step}], Loss: {loss.item():.4f}')

model.eval()
with torch.no_grad():
  correct = 0
  total = 0
  for images, labels in test_loader:
    images = images.to(device)
    labels = labels.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct +=  (predicted == labels).sum().item()

  print('Akurasi Model pada {} gambar rontgen uji: {:.2f} %'.format(total, 100 * correct / total))

Akurasi Model pada 40 gambar rontgen uji: 97.50 %
